# Encoder Model Fine-Tuning and Comparison on SQuAD v2

This notebook fine-tunes multiple encoder models (BERT, RoBERTa, DistilBERT, etc.) on the SQuAD v2 dataset and compares their performance using statistical analysis.

## What This Notebook Does:

1. **Loads SQuAD v2 Dataset**: Uses the `rajpurkar/squad_v2` dataset from HuggingFace
2. **Fine-tunes Multiple Encoder Models**: Trains several encoder models for question answering
3. **Evaluates Performance**: Uses F1 score and Exact Match (EM) metrics
4. **Statistical Comparison**: Performs MANOVA and pairwise comparisons to determine which models perform significantly better
5. **Saves Results**: Tracks all metrics and model checkpoints for analysis


In [ ]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Should show the number of GPUs
print(torch.cuda.get_device_name(0))  # Should show the GPU model


True
1
NVIDIA RTX 5000 Ada Generation


## Hyperparameter convention (experimental control)

Fine-tuning settings follow literature-based convention: **small** (<100M params), **medium** (~110M), **large** (300M+). Larger models use slightly lower learning rates for stable convergence. We use linear warmup (~10% of steps) and AdamW per Mosbach et al.

- **Devlin et al., 2019.** BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. NAACL-HLT. https://aclanthology.org/N19-1423/
- **Mosbach et al., 2021.** On the Stability of Fine-tuning BERT: Misconceptions, Explanations, and Strong Baselines. OpenReview. https://openreview.net/forum?id=nzpLWnVAyah

In [1]:
import transformers
import torch
import os
import json
from huggingface_hub import login
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)
from datasets import load_dataset
import logging
import time
from datetime import timedelta, datetime
import pandas as pd
from dotenv import load_dotenv
import numpy as np
import evaluate
from tqdm import tqdm

# Load environment variables
load_dotenv(dotenv_path="../../.env")

# Experiment configuration (hyperparams by size: Devlin et al. 2019, Mosbach et al. 2021)
# Small <100M, Base/Medium ~110M, Large 300M+. Larger models use slightly lower LR.
run_id = "Encoder_RUN01"  # <- Change this manually for each experiment
max_length = 384  # <- Maximum sequence length
doc_stride = 128  # <- Stride for long documents

# Hyperparameters by size tier (experimental control)
TIER_LEARNING_RATE = {"small": 5e-5, "medium": 3e-5, "large": 2e-5}
TIER_BATCH_SIZE = {"small": 16, "medium": 16, "large": 8}
TIER_EPOCHS = {"small": 3, "medium": 3, "large": 3}

# Model name -> size tier (small <100M, medium ~110M, large 300M+)
MODEL_TIER = {
    "bert-base-uncased": "medium",
    "roberta-base": "medium",
    "distilbert-base-uncased": "small",
    "albert-base-v2": "small",
    "microsoft/deberta-v3-base": "medium",
}

# Models to compare (add new entries to MODEL_TIER when adding models)
models_to_test = [
    "bert-base-uncased",
    "roberta-base",
    "distilbert-base-uncased",
    "albert-base-v2",
    "microsoft/deberta-v3-base",
]


In [2]:
import logging
logging.basicConfig(
    filename=f'{run_id}_training.log', 
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.info(f"Run ID: {run_id}")
logging.info(f"Models to test: {models_to_test}")
logging.info(f"Training config: per-tier LR={TIER_LEARNING_RATE}, batch={TIER_BATCH_SIZE}, epochs={TIER_EPOCHS}")


In [3]:
# Setting huggingface token
login(token=os.getenv("HF_API_KEY"))

# Set cache directories (adjust paths as needed)
os.environ["HF_HOME"] = "D:/huggingface_cache" 
os.environ["TRANSFORMERS_CACHE"] = "D:/huggingface_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "D:/huggingface_cache"

print("HF_HOME:", os.getenv("HF_HOME"))
print("TRANSFORMERS_CACHE:", os.getenv("TRANSFORMERS_CACHE"))
print("HUGGINGFACE_HUB_CACHE:", os.getenv("HUGGINGFACE_HUB_CACHE"))

logging.info(f"HF_HOME: {os.getenv('HF_HOME')}")
logging.info(f"TRANSFORMERS_CACHE: {os.getenv('TRANSFORMERS_CACHE')}")
logging.info(f"HUGGINGFACE_HUB_CACHE: {os.getenv('HUGGINGFACE_HUB_CACHE')}")


HF_HOME: D:/huggingface_cache
TRANSFORMERS_CACHE: D:/huggingface_cache
HUGGINGFACE_HUB_CACHE: D:/huggingface_cache


## Load SQuAD v2 Dataset


In [4]:
# Load SQuAD v2 dataset
print("Loading SQuAD v2 dataset...")
squad_dataset = load_dataset("rajpurkar/squad_v2")

print(f"Train set size: {len(squad_dataset['train'])}")
print(f"Validation set size: {len(squad_dataset['validation'])}")
print(f"\nSample data:")
print(squad_dataset['train'][0])

logging.info(f"Loaded SQuAD v2: train={len(squad_dataset['train'])}, val={len(squad_dataset['validation'])}")


Loading SQuAD v2 dataset...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

c:\Users\ctngweru\AppData\Local\anaconda3\envs\lm-forge\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ctngweru\.cache\huggingface\hub\datasets--rajpurkar--squad_v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling 

validation-00000-of-00001.parquet:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Train set size: 130319
Validation set size: 11873

Sample data:
{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}


## Preprocessing Functions


In [5]:
def preprocess_function(examples, tokenizer):
    """
    Preprocess SQuAD v2 examples for question answering.
    Handles unanswerable questions (where answers are None).
    """
    questions = [q.strip() for q in examples["question"]]
    contexts = examples["context"]
    
    # Tokenize
    tokenized_examples = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    # Map example IDs to features
    sample_mapping = tokenized_examples["overflow_to_sample_mapping"]
    example_ids = []
    for sample_idx in sample_mapping:
        example_ids.append(examples["id"][sample_idx])
    
    # Get offset mappings (keep for evaluation)
    offset_mapping = tokenized_examples.pop("offset_mapping")
    
    # Handle answers
    start_positions = []
    end_positions = []
    
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        answers = examples["answers"][sample_idx]
        
        # If no answer, set positions to 0 (CLS token)
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Find answer span
            answer_start = answers["answer_start"][0]
            answer_text = answers["text"][0]
            
            # Find token positions
            start_char = answer_start
            end_char = answer_start + len(answer_text)
            
            sequence_ids = tokenized_examples.sequence_ids(i)
            
            # Find start and end token positions
            idx = 0
            while idx < len(offsets) and sequence_ids[idx] != 1:
                idx += 1
            context_start = idx
            
            while idx < len(offsets) and sequence_ids[idx] == 1:
                idx += 1
            context_end = idx - 1
            
            # Find answer span in tokens
            start_token = context_start
            end_token = context_end
            
            for token_idx, (token_start, token_end) in enumerate(offsets[context_start:context_end+1], start=context_start):
                if token_start <= start_char < token_end:
                    start_token = token_idx
                if token_start < end_char <= token_end:
                    end_token = token_idx
            
            start_positions.append(start_token)
            end_positions.append(end_token)
    
    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    tokenized_examples["offset_mapping"] = offset_mapping  # Keep for evaluation
    tokenized_examples["example_id"] = example_ids  # Store example IDs
    
    return tokenized_examples


In [6]:
def postprocess_qa_predictions(examples, features, predictions, tokenizer, n_best_size=20, max_answer_length=30):
    """
    Post-process predictions to extract answers from the model outputs.
    """
    all_start_logits, all_end_logits = predictions
    
    # Build a map from example to its features
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = {}
    
    for i, feature in enumerate(features):
        example_id = feature["example_id"]
        if example_id not in features_per_example:
            features_per_example[example_id] = []
        features_per_example[example_id].append(i)
    
    # Get predictions
    final_predictions = {}
    
    for example_id, example_index in example_id_to_index.items():
        example = examples[example_index]
        context = example["context"]
        
        # Get all features for this example
        feature_indices = features_per_example[example_id]
        
        min_null_score = None
        valid_answers = []
        
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            
            # Get offset mapping for this feature
            offset_mapping = features[feature_index]["offset_mapping"]
            
            # Get top predictions
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
            
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offset_mapping):
                        continue
                    if end_index >= len(offset_mapping):
                        continue
                    if start_index > end_index:
                        continue
                    if end_index - start_index + 1 > max_answer_length:
                        continue
                    
                    # Get character offsets
                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    
                    # Extract answer text from context
                    answer_text = context[start_char:end_char]
                    
                    valid_answers.append({
                        "text": answer_text,
                        "score": start_logits[start_index] + end_logits[end_index],
                    })
            
            # Calculate null score (for unanswerable questions)
            null_score = start_logits[0] + end_logits[0]
            if min_null_score is None or null_score < min_null_score:
                min_null_score = null_score
        
        # Get best answer
        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
            if best_answer["score"] > min_null_score:
                final_predictions[example_id] = best_answer["text"]
            else:
                final_predictions[example_id] = ""
        else:
            final_predictions[example_id] = ""
    
    return final_predictions


In [7]:
def compute_metrics(eval_pred, tokenizer, examples, features):
    """
    Compute F1 and Exact Match metrics for question answering.
    """
    predictions, labels = eval_pred
    start_logits, end_logits = predictions
    
    # Post-process predictions
    predictions_dict = postprocess_qa_predictions(
        examples, features, (start_logits, end_logits), tokenizer
    )
    
    # Format predictions for metric computation
    formatted_predictions = [
        {"id": k, "prediction_text": v} for k, v in predictions_dict.items()
    ]
    
    # Format references
    references = [
        {"id": ex["id"], "answers": ex["answers"]} for ex in examples
    ]
    
    # Load SQuAD metric
    squad_metric = evaluate.load("squad_v2")
    
    # Compute metrics
    results = squad_metric.compute(predictions=formatted_predictions, references=references)
    
    return results


In [8]:
# Create output directory
if not os.path.exists(run_id):
    os.makedirs(run_id)
    os.makedirs(f"{run_id}/checkpoints")
    os.makedirs(f"{run_id}/results")

# Initialize results DataFrame
results_df = pd.DataFrame(columns=[
    "model_name",
    "f1_score",
    "exact_match",
    "total_samples",
    "training_time",
    "num_epochs",
    "batch_size",
    "learning_rate",
    "max_length",
    "model_size_mb",
    "timestamp"
])

# Checkpoint tracker
checkpoint_path = f"{run_id}/training_tracker.csv"
if os.path.exists(checkpoint_path):
    tracker_df = pd.read_csv(checkpoint_path)
else:
    tracker_df = pd.DataFrame(columns=["model_name", "completed", "f1_score", "exact_match"])
    for model_name in models_to_test:
        tracker_df.loc[len(tracker_df)] = [model_name, False, None, None]
    tracker_df.to_csv(checkpoint_path, index=False)

print(f"Results will be saved to: {run_id}/")
logging.info(f"Output directory: {run_id}/")


Results will be saved to: Encoder_RUN01/


## Fine-Tune Each Model


In [9]:
# Fine-tune each model
for model_name in models_to_test:
    print(f"\n{'='*80}")
    print(f"Training model: {model_name}")
    print(f"{'='*80}")
    logging.info(f"Starting training for model: {model_name}")
    
    # Check if already completed
    row_match = tracker_df["model_name"] == model_name
    if tracker_df.loc[row_match, "completed"].any():
        print(f"Skipping {model_name} (already completed)")
        logging.info(f"Skipping {model_name} (already completed)")
        continue
    
    try:
        # Load tokenizer and model
        print(f"Loading {model_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        
        # Move to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        
        # Calculate model size
        model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)
        
        # Preprocess datasets
        print("Preprocessing training data...")
        train_dataset = squad_dataset["train"].map(
            lambda x: preprocess_function(x, tokenizer),
            batched=True,
            remove_columns=squad_dataset["train"].column_names,
        )
        
        print("Preprocessing validation data...")
        val_dataset = squad_dataset["validation"].map(
            lambda x: preprocess_function(x, tokenizer),
            batched=True,
            remove_columns=squad_dataset["validation"].column_names,
        )
        
        # Prepare validation examples and features for evaluation
        val_examples = squad_dataset["validation"]
        val_features = val_dataset
        
        # Data collator
        data_collator = DefaultDataCollator()
        
        # Per-model hyperparams by size tier (Devlin et al. / Mosbach et al.)
        tier = MODEL_TIER.get(model_name, "medium")
        learning_rate = TIER_LEARNING_RATE[tier]
        batch_size = TIER_BATCH_SIZE[tier]
        num_epochs = TIER_EPOCHS[tier]
        
        # Training arguments (warmup_ratio ~10% per Mosbach et al. 2021)
        training_args = TrainingArguments(
            output_dir=f"{run_id}/checkpoints/{model_name.replace('/', '_')}",
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=learning_rate,
            weight_decay=0.01,
            warmup_ratio=0.1,
            logging_dir=f"{run_id}/logs/{model_name.replace('/', '_')}",
            logging_steps=100,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            save_total_limit=2,
            report_to="none",  # Set to "wandb" if you want to use Weights & Biases
        )
        
        # Custom compute_metrics wrapper
        def compute_metrics_wrapper(eval_pred):
            return compute_metrics(eval_pred, tokenizer, val_examples, val_features)
        
        # Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics_wrapper,
        )
        
        # Train
        start_time = time.time()
        print("Starting training...")
        trainer.train()
        training_time = timedelta(seconds=time.time() - start_time)
        
        # Evaluate
        print("Evaluating model...")
        eval_results = trainer.evaluate()
        
        f1_score = eval_results.get("eval_f1", 0.0)
        exact_match = eval_results.get("eval_exact_match", 0.0)
        
        print(f"\nResults for {model_name}:")
        print(f"  F1 Score: {f1_score:.4f}")
        print(f"  Exact Match: {exact_match:.4f}")
        print(f"  Training Time: {training_time}")
        
        # Save results
        results_df.loc[len(results_df)] = [
            model_name,
            f1_score,
            exact_match,
            len(squad_dataset["validation"]),
            str(training_time),
            num_epochs,
            batch_size,
            learning_rate,
            max_length,
            model_size_mb,
            datetime.now().isoformat(),
        ]
        
        # Update tracker
        tracker_df.loc[row_match, "completed"] = True
        tracker_df.loc[row_match, "f1_score"] = f1_score
        tracker_df.loc[row_match, "exact_match"] = exact_match
        tracker_df.to_csv(checkpoint_path, index=False)
        
        # Save results CSV
        results_df.to_csv(f"{run_id}/results/training_results.csv", index=False)
        
        logging.info(f"Completed {model_name}: F1={f1_score:.4f}, EM={exact_match:.4f}, Time={training_time}")
        
        # Clear memory
        del model, tokenizer, trainer
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        logging.error(f"Error training {model_name}: {e}", exc_info=True)
        continue

print(f"\n{'='*80}")
print("All models trained!")
print(f"{'='*80}")



Training model: bert-base-uncased
Loading bert-base-uncased...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\ctngweru\AppData\Local\anaconda3\envs\lm-forge\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ctngweru\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

C:\Users\ctngweru\AppData\Local\Temp\ipykernel_45156\4107009700.py:74: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Load Results for Analysis


In [ ]:
# Load results
results_path = f"{run_id}/results/training_results.csv"
if os.path.exists(results_path):
    scores_df = pd.read_csv(results_path)
    print("Loaded results:")
    print(scores_df)
else:
    print("No results file found. Please run training first.")
    scores_df = pd.DataFrame()


## Statistical Analysis

### MANOVA Assumptions
- Multivariate Normality: dependent variables should be jointly normally distributed
- Homogeneity of variance-covariance matrices
- Independence of observations
- No multicollinearity
- Linearity
- Outliers


In [ ]:
from scipy.stats import chi2, probplot
from sklearn.covariance import MinCovDet
import matplotlib.pyplot as plt
import pingouin as pg
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from statsmodels.multivariate.manova import MANOVA
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import permanova, DistanceMatrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

if len(scores_df) > 0:
    # Define dependent variables
    dv_columns = ["f1_score", "exact_match"]
    
    # Select only the dependent variables
    X = scores_df[dv_columns].dropna()
    
    print(f"Number of models: {len(scores_df)}")
    print(f"Dependent variables: {dv_columns}")


In [ ]:
# Multivariate Normality Test
if len(scores_df) > 0:
    # Compute robust Mahalanobis distances
    robust_cov = MinCovDet().fit(X)
    mahalanobis_distances = robust_cov.mahalanobis(X)
    
    # Chi-squared Q-Q plot
    probplot(mahalanobis_distances, dist="chi2", sparams=(len(dv_columns),), plot=plt)
    plt.title("Q-Q Plot of Mahalanobis Distances")
    plt.show()
    
    # Formal multivariate normality test (Henze-Zirkler's)
    if len(scores_df) >= 4:  # Need at least 4 samples for the test
        normality_test = pg.multivariate_normality(scores_df[dv_columns], alpha=0.05)
        print("Henze-Zirkler Test:\n", normality_test)
    else:
        print("Not enough samples for multivariate normality test (need at least 4)")


In [ ]:
# Outlier Detection
if len(scores_df) > 0:
    threshold = chi2.ppf(0.99, df=len(dv_columns))
    outliers = mahalanobis_distances > threshold
    
    print(f"Outlier count: {np.sum(outliers)}")
    if np.sum(outliers) > 0:
        outlier_indices = np.where(outliers)[0]
        outlier_rows = scores_df.iloc[outlier_indices]
        print("Outlier rows:")
        print(outlier_rows[["model_name"] + dv_columns])


In [ ]:
# Check for multicollinearity
if len(scores_df) > 0:
    X_with_const = add_constant(X)
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_with_const.columns
    vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) 
                        for i in range(X_with_const.shape[1])]
    print("Variance Inflation Factors:")
    print(vif_data)


In [ ]:
# Correlation between dependent variables
if len(scores_df) > 0:
    plt.figure(figsize=(8, 6))
    sns.heatmap(scores_df[dv_columns].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Correlation between F1 Score and Exact Match")
    plt.tight_layout()
    plt.show()


In [ ]:
# MANOVA (if we have multiple models)
if len(scores_df) > 0 and len(scores_df) >= 3:
    scores_df["group"] = scores_df["model_name"]
    df_clean = scores_df.dropna(subset=dv_columns + ["group"])
    
    # Create MANOVA formula
    formula = " + ".join(dv_columns) + " ~ group"
    
    # Fit MANOVA
    maov = MANOVA.from_formula(formula, data=df_clean)
    print("MANOVA Results:")
    print(maov.mv_test())
else:
    print("Not enough models for MANOVA (need at least 3 models)")


In [ ]:
# Type III ANOVA for each dependent variable
if len(scores_df) > 0 and len(scores_df) >= 3:
    def type_3(formula, data):
        model = ols(formula, data=data).fit()
        anova_results = anova_lm(model, typ=3)
        return anova_results
    
    results = []
    for column in dv_columns:
        formula = f"{column} ~ group"
        anova_results = type_3(formula, df_clean)
        print(f"\nANOVA results for {column}:\n", anova_results)
        results.append((column, anova_results))


In [ ]:
# Pairwise comparisons
from itertools import combinations

if len(scores_df) > 0 and len(scores_df) >= 2:
    pairwise_results = []
    
    for dv in dv_columns:
        for model1, model2 in combinations(scores_df["model_name"].unique(), 2):
            # Since we have one score per model, we'll compute the difference
            val1 = scores_df[scores_df["model_name"] == model1][dv].values[0]
            val2 = scores_df[scores_df["model_name"] == model2][dv].values[0]
            
            diff = val1 - val2
            
            pairwise_results.append({
                "Metric": dv,
                "Model 1": model1,
                "Model 2": model2,
                "Difference": diff,
                "Model 1 Score": val1,
                "Model 2 Score": val2,
            })
    
    pairwise_df = pd.DataFrame(pairwise_results)
    print("Pairwise Comparisons:")
    print(pairwise_df)
    pairwise_df.to_csv(f"{run_id}/results/pairwise_comparisons.csv", index=False)
else:
    print("Not enough models for pairwise comparisons")


In [ ]:
# Visualization: Model Comparison
if len(scores_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # F1 Score comparison
    axes[0].barh(scores_df["model_name"], scores_df["f1_score"])
    axes[0].set_xlabel("F1 Score")
    axes[0].set_title("F1 Score by Model")
    axes[0].grid(axis='x', alpha=0.3)
    
    # Exact Match comparison
    axes[1].barh(scores_df["model_name"], scores_df["exact_match"])
    axes[1].set_xlabel("Exact Match Score")
    axes[1].set_title("Exact Match by Model")
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{run_id}/results/model_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Combined scatter plot
    plt.figure(figsize=(10, 6))
    for idx, row in scores_df.iterrows():
        plt.scatter(row["f1_score"], row["exact_match"], s=200, alpha=0.6)
        plt.annotate(row["model_name"], 
                    (row["f1_score"], row["exact_match"]),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    plt.xlabel("F1 Score")
    plt.ylabel("Exact Match")
    plt.title("Model Performance: F1 vs Exact Match")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{run_id}/results/f1_vs_em_scatter.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Summary statistics
if len(scores_df) > 0:
    print("\nSummary Statistics:")
    print("="*80)
    print(scores_df[["model_name", "f1_score", "exact_match", "model_size_mb", "training_time"]].to_string(index=False))
    
    print("\n\nBest Models:")
    print("="*80)
    best_f1 = scores_df.loc[scores_df["f1_score"].idxmax()]
    best_em = scores_df.loc[scores_df["exact_match"].idxmax()]
    
    print(f"Best F1 Score: {best_f1['model_name']} ({best_f1['f1_score']:.4f})")
    print(f"Best Exact Match: {best_em['model_name']} ({best_em['exact_match']:.4f})")
    
    # Save summary
    summary = {
        "total_models": len(scores_df),
        "best_f1_model": best_f1['model_name'],
        "best_f1_score": float(best_f1['f1_score']),
        "best_em_model": best_em['model_name'],
        "best_em_score": float(best_em['exact_match']),
        "mean_f1": float(scores_df["f1_score"].mean()),
        "mean_em": float(scores_df["exact_match"].mean()),
        "std_f1": float(scores_df["f1_score"].std()),
        "std_em": float(scores_df["exact_match"].std()),
    }
    
    with open(f"{run_id}/results/summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    
    print(f"\nSummary saved to: {run_id}/results/summary.json")
